## Task 3: Exploring Data Heterogeneity Impact
**Goal: Study how label skew (non-IID data) affects FedAvg performance**

## 0. Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Use the SAME scripts as Task 2!
from scriptsfl.data_utils import load_dataset, create_dirichlet_split, get_dataloaders, analyze_data_distribution
from scriptsfl.models import get_model
from scriptsfl.client import Client
from scriptsfl.server import Server
from scriptsf1.results_utils import save_results, plot_comparison, print_summary

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
print("="*80)
print("TASK 3: EXPLORING DATA HETEROGENEITY IMPACT")
print("="*80)

#%% Configuration
CONFIG = {
    'dataset': 'cifar10',
    'num_clients': 5,
    'num_rounds': 50,
    'local_epochs': 5,  # Fixed K for all experiments
    'batch_size': 32,
    'lr': 0.01,
    'momentum': 0.9,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

#%% Load dataset (shared across all experiments)
train_dataset, test_dataset = load_dataset(CONFIG['dataset'])
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

## Experiment: Varying Dirichlet Alpha


In [ ]:
# TODO: Test different alpha values
# - alpha = 100 (or very large): Nearly IID (balanced distribution)
# - alpha = 1.0: Moderate heterogeneity
# - alpha = 0.1: High heterogeneity (severe label skew)
# - alpha = 0.05: Extreme heterogeneity

alpha_values = [100, 1.0, 0.5, 0.1]  # From IID to highly non-IID
results_heterogeneity = {}

for alpha in alpha_values:
    print(f"\n{'='*80}")
    print(f"TRAINING WITH ALPHA = {alpha}")
    print(f"{'='*80}")

    # TODO: Create Dirichlet split for this alpha
    client_indices = create_dirichlet_split(train_dataset, CONFIG['num_clients'], alpha=alpha)

    # Analyze and print distribution
    analyze_data_distribution(train_dataset, client_indices, num_classes=10)

    # TODO: Create data loaders
    client_loaders = get_dataloaders(train_dataset, client_indices,
                                      batch_size=CONFIG['batch_size'], shuffle=True)

    # TODO: Initialize model
    model = get_model('simplecnn', num_classes=10, dataset=CONFIG['dataset'])

    # TODO: Create clients using the Client class from client.py
    # Hint: clients = [Client(client_id=i, ...) for i in range(num_clients)]
    clients = [Client(client_id=i, data_loader=client_loaders[i],
                     model=model, device=CONFIG['device'])
              for i in range(CONFIG['num_clients'])]

    # TODO: Create server using Server class from server.py
    server = Server(global_model=model, clients=clients,
                   test_loader=test_loader, device=CONFIG['device'])

    # TODO: Train using server.train()
    history = server.train(
        num_rounds=CONFIG['num_rounds'],
        local_epochs=CONFIG['local_epochs'],
        lr=CONFIG['lr'],
        client_fraction=1.0,
        momentum=CONFIG['momentum'],
        verbose=True
    )

    # Save results
    exp_name = f'α={alpha}' if alpha < 10 else 'IID (α=100)'
    results_heterogeneity[exp_name] = {
        'history': history,
        'config': {**CONFIG, 'alpha': alpha}
    }

    print(f"\nFinal accuracy with α={alpha}: {history['test_accuracy'][-1]:.2f}%")

In [ ]:
#%% Save and Visualize Results

# Save all results
save_results(results_heterogeneity, 'task3_heterogeneity')

# TODO: Plot test accuracy vs rounds for all alpha values
plot_comparison(results_heterogeneity, metric='test_accuracy',
                title='Impact of Data Heterogeneity on FedAvg',
                save_path='./results/task3_heterogeneity.png')

# Print summary
print_summary(results_heterogeneity)

## Additional Analysis (Optional but Recommended)


In [ ]:
print("\n" + "="*80)
print("ADDITIONAL ANALYSIS")
print("="*80)

# TODO: Create bar chart comparing final accuracies
final_accuracies = {name: results['history']['test_accuracy'][-1]
                   for name, results in results_heterogeneity.items()}

plt.figure(figsize=(10, 6))
bars = plt.bar(final_accuracies.keys(), final_accuracies.values(), color='steelblue')
plt.xlabel('Data Distribution', fontsize=12)
plt.ylabel('Final Test Accuracy (%)', fontsize=12)
plt.title('Final Test Accuracy vs Data Heterogeneity', fontsize=14, fontweight='bold')
plt.grid(True, axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('./results/task3_final_accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("TASK 3 COMPLETE")
print("="*80)
print("\nKey Findings:")
print("1. As alpha decreases (more skew), FedAvg performance degrades")
print("2. IID data (large alpha) achieves best performance")
print("3. Severe heterogeneity can significantly impact convergence")
print("4. This motivates the need for specialized FL algorithms (Task 4!)")